## Electron repulsion integrals (ERIs)

In [56]:
import sympy as sp
alpha, beta = sp.symbols('alpha beta', real = True, positive = True)
Ax, Ay, Az = sp.symbols('Ax Ay Az', real = True)
Bx, By, Bz = sp.symbols('Bx By Bz', real = True)
gamma, delta = sp.symbols('gamma delta', real = True, positive = True)
Cx, Cy, Cz = sp.symbols('Cx Cy Cz', real = True)
Dx, Dy, Dz = sp.symbols('Dx Dy Dz', real = True)

p = alpha + beta
q = gamma + delta
pq = p + q
rho = p*q/pq

Px = (alpha*Ax + beta*Bx)/p
Py = (alpha*Ay + beta*By)/p
Pz = (alpha*Az + beta*Bz)/p

Qx = (gamma*Cx + delta*Dx)/q
Qy = (gamma*Cy + delta*Dy)/q
Qz = (gamma*Cz + delta*Dz)/q

RAB = ((Ax-Bx)**2+(Ay-By)**2+(Az-Bz)**2)
RCD = ((Cx-Dx)**2+(Cy-Dy)**2+(Cz-Dz)**2)
RPQ = ((Px-Qx)**2+(Py-Qy)**2+(Pz-Qz)**2)

In [57]:
class Boys(sp.Function):
    nargs = 2

    def fdiff(self, argindex=1):
        if argindex == 2:
            return -Boys(self.args[0] + 1, self.args[1])
        raise ValueError(argindex)

In [58]:
G00 = (2*sp.pi/rho)*(sp.pi/(p+q))**(3/2)*sp.exp(-(alpha*beta/p)*RAB)*sp.exp(-(gamma*delta/q)*RCD)*Boys(0, rho*RPQ)
G00

2*pi**2.5*exp(-alpha*beta*((Ax - Bx)**2 + (Ay - By)**2 + (Az - Bz)**2)/(alpha + beta))*exp(-delta*gamma*((Cx - Dx)**2 + (Cy - Dy)**2 + (Cz - Dz)**2)/(delta + gamma))*Boys(0, (alpha + beta)*(delta + gamma)*((-(Cx*gamma + Dx*delta)/(delta + gamma) + (Ax*alpha + Bx*beta)/(alpha + beta))**2 + (-(Cy*gamma + Dy*delta)/(delta + gamma) + (Ay*alpha + By*beta)/(alpha + beta))**2 + (-(Cz*gamma + Dz*delta)/(delta + gamma) + (Az*alpha + Bz*beta)/(alpha + beta))**2)/(alpha + beta + delta + gamma))/((alpha + beta)*(delta + gamma)*(alpha + beta + delta + gamma)**0.5)

In [59]:
from functools import lru_cache

@lru_cache(maxsize = None)
def get_ckn(k: int, n: int, p):
    if k<0 or k>n:
        return sp.Integer(0)
    if n == 0:
        return sp.Integer(1) if k==0 else sp.Integer(0)
    return get_ckn(k-1, n-1, p)/(2*p) + (k+1)*get_ckn(k+1, n-1, p)

In [60]:
def build_eri_derivative_table(Gp, Lmax, Avars, Bvars, Cvars, Dvars):
    Ax, Ay, Az = Avars
    Bx, By, Bz = Bvars
    Cx, Cy, Cz = Cvars
    Dx, Dy, Dz = Dvars

    all_idx = [
        (i, j, L - i - j)
        for L in range(Lmax + 1)
        for i in range(L + 1)
        for j in range(L + 1 - i)
    ]

    @lru_cache(maxsize=None)
    def G(i, j, k, l, m, n, o, p, q, r, s, t):
        if (i, j, k, l, m, n, o, p, q, r, s, t) == (0,) * 12:
            return Gp

        if i > 0: return sp.diff(G(i - 1, j, k, l, m, n, o, p, q, r, s, t), Ax)
        if j > 0: return sp.diff(G(i, j - 1, k, l, m, n, o, p, q, r, s, t), Ay)
        if k > 0: return sp.diff(G(i, j, k - 1, l, m, n, o, p, q, r, s, t), Az)

        if l > 0: return sp.diff(G(i, j, k, l - 1, m, n, o, p, q, r, s, t), Bx)
        if m > 0: return sp.diff(G(i, j, k, l, m - 1, n, o, p, q, r, s, t), By)
        if n > 0: return sp.diff(G(i, j, k, l, m, n - 1, o, p, q, r, s, t), Bz)

        if o > 0: return sp.diff(G(i, j, k, l, m, n, o - 1, p, q, r, s, t), Cx)
        if p > 0: return sp.diff(G(i, j, k, l, m, n, o, p - 1, q, r, s, t), Cy)
        if q > 0: return sp.diff(G(i, j, k, l, m, n, o, p, q - 1, r, s, t), Cz)

        if r > 0: return sp.diff(G(i, j, k, l, m, n, o, p, q, r - 1, s, t), Dx)
        if s > 0: return sp.diff(G(i, j, k, l, m, n, o, p, q, r, s - 1, t), Dy)
        return sp.diff(G(i, j, k, l, m, n, o, p, q, r, s, t - 1), Dz)

    derivatives = {}
    for a in all_idx:
        for b in all_idx:
            for c in all_idx:
                for d in all_idx:
                    key = a + b + c + d
                    derivatives[key] = G(*key)
    return derivatives


In [61]:
Lmax = 1

In [62]:
derivatives_dict =  build_eri_derivative_table(G00, Lmax=1, Avars=(Ax, Ay, Az), Bvars=(Bx, By, Bz), Cvars=(Cx, Cy, Cz), Dvars=(Dx, Dy, Dz))

In [63]:
from itertools import product

def nonzero_ckn(order, exponent):
    out = []
    for idx in range(order + 1):
        coeff = get_ckn(idx, order, exponent)
        if coeff != 0:
            out.append((idx, coeff))
    return out


def get_twoel(i, j, k, l, m, n, o, p, q, r, s, t, derivatives_eri):
    orders = (i, j, k, l, m, n, o, p, q, r, s, t)
    exponents = (
        alpha, alpha, alpha,
        beta, beta, beta,
        gamma, gamma, gamma,
        delta, delta, delta,
    )

    coeff_axes = tuple(
        nonzero_ckn(order, exponent)
        for order, exponent in zip(orders, exponents)
    )

    eri = sp.Integer(0)
    for term in product(*coeff_axes):
        idx = tuple(axis_idx for axis_idx, _ in term)
        coeff = sp.prod(axis_coeff for _, axis_coeff in term)
        eri += coeff * derivatives_eri[idx]

    return sp.factor_terms(eri)


In [64]:
def build_eri_dict(derivatives_eri):
    for key in sorted(derivatives_eri):
        expr = get_twoel(*key, derivatives_eri=derivatives_eri)
        eri_dict[key] = expr
    return eri_dict


In [65]:
eri_dict = build_eri_dict(derivatives_dict)

In [66]:
rho, RPQ2 = sp.symbols("rho RPQ2", positive=True, real=True)

subsdict = {
    # basic Gaussian sums
    alpha + beta: P,
    gamma + delta: Q,

    # A-B and C-D differences
    Ax - Bx: ABx,
    Ay - By: ABy,
    Az - Bz: ABz,
    Cx - Dx: CDx,
    Cy - Dy: CDy,
    Cz - Dz: CDz,

    # squared distances
    (Ax - Bx)**2 + (Ay - By)**2 + (Az - Bz)**2: RAB2,
    (Cx - Dx)**2 + (Cy - Dy)**2 + (Cz - Dz)**2: RCD2,

    # product centers
    (alpha*Ax + beta*Bx)/(alpha + beta): Px,
    (alpha*Ay + beta*By)/(alpha + beta): Py,
    (alpha*Az + beta*Bz)/(alpha + beta): Pz,

    (gamma*Cx + delta*Dx)/(gamma + delta): Qx,
    (gamma*Cy + delta*Dy)/(gamma + delta): Qy,
    (gamma*Cz + delta*Dz)/(gamma + delta): Qz,

    # P-Q components
    (alpha*Ax + beta*Bx)/(alpha + beta) - (gamma*Cx + delta*Dx)/(gamma + delta): PQx,
    (alpha*Ay + beta*By)/(alpha + beta) - (gamma*Cy + delta*Dy)/(gamma + delta): PQy,
    (alpha*Az + beta*Bz)/(alpha + beta) - (gamma*Cz + delta*Dz)/(gamma + delta): PQz,

    -(gamma*Cx + delta*Dx)/(gamma + delta) + (alpha*Ax + beta*Bx)/(alpha + beta): PQx,
    -(gamma*Cy + delta*Dy)/(gamma + delta) + (alpha*Ay + beta*By)/(alpha + beta): PQy,
    -(gamma*Cz + delta*Dz)/(gamma + delta) + (alpha*Az + beta*Bz)/(alpha + beta): PQz,

    # |P-Q|^2
    ((alpha*Ax + beta*Bx)/(alpha + beta) - (gamma*Cx + delta*Dx)/(gamma + delta))**2
    + ((alpha*Ay + beta*By)/(alpha + beta) - (gamma*Cy + delta*Dy)/(gamma + delta))**2
    + ((alpha*Az + beta*Bz)/(alpha + beta) - (gamma*Cz + delta*Dz)/(gamma + delta))**2: RPQ2,

    # rho = pq/(p+q)
    (alpha + beta)*(gamma + delta)/(alpha + beta + gamma + delta): rho,

    # Gaussian prefactors
    sp.exp(-alpha*beta*((Ax - Bx)**2 + (Ay - By)**2 + (Az - Bz)**2)/(alpha + beta)): KAB,
    sp.exp(-gamma*delta*((Cx - Dx)**2 + (Cy - Dy)**2 + (Cz - Dz)**2)/(gamma + delta)): KCD,
}

In [67]:
for key, value in eri_dict.items():
    eri_dict[key] = value.subs(subsdict, simultaneous=True)


In [68]:
for key, value in eri_dict.items():
    print(f"{key}: {value}")

(0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0): 2*pi**2.5*KAB*KCD*Boys(0, RPQ2*rho)/(P*Q*(P + Q)**0.5)
(0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1): 2*pi**2.5*KAB*KCD*(CDz*gamma*Boys(0, RPQ2*rho)/(P*Q*(P + Q)**0.5) + PQz*Boys(1, RPQ2*rho)/(P + Q)**1.5)/Q
(0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0): 2*pi**2.5*KAB*KCD*(CDy*gamma*Boys(0, RPQ2*rho)/(P*Q*(P + Q)**0.5) + PQy*Boys(1, RPQ2*rho)/(P + Q)**1.5)/Q
(0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0): 2*pi**2.5*KAB*KCD*(CDx*gamma*Boys(0, RPQ2*rho)/(P*Q*(P + Q)**0.5) + PQx*Boys(1, RPQ2*rho)/(P + Q)**1.5)/Q
(0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0): 2*pi**2.5*KAB*KCD*(-CDz*delta*Boys(0, RPQ2*rho)/(P*Q*(P + Q)**0.5) + PQz*Boys(1, RPQ2*rho)/(P + Q)**1.5)/Q
(0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1): pi**2.5*KAB*KCD*(-2*CDz**2*delta*gamma*Boys(0, RPQ2*rho)/(P*Q**2*(P + Q)**0.5) - 2*CDz*PQz*delta*Boys(1, RPQ2*rho)/(Q*(P + Q)**1.5) + 2*CDz*PQz*gamma*Boys(1, RPQ2*rho)/(Q*(P + Q)**1.5) + 2*P*PQz**2*Boys(2, RPQ2*rho)/(P + Q)**2.5 - Boys(1, RPQ2*rho)/(Q*(P + Q)**1.5) + Boys(0, RPQ2*rho)/(P*Q*(

In [69]:
args = set()

for expr in eri_dict.values():
    args.update(expr.free_symbols)

args = sorted(args, key = lambda x: x.name)
args = ["i","j","k","l","m","n","o","p","q","r","s","t"] + list(args)
args = ", ".join(str(arg) for arg in args)
args

'i, j, k, l, m, n, o, p, q, r, s, t, ABx, ABy, ABz, CDx, CDy, CDz, KAB, KCD, P, PQx, PQy, PQz, Q, RPQ2, alpha, beta, delta, gamma, rho'

In [70]:
from pathlib import Path
from sympy.printing.numpy import NumPyPrinter, _known_functions_numpy, _known_constants_numpy

class TheochemNumPyPrinter(NumPyPrinter):
    def print_boys(self, expr):
        n, t = expr.args
        return f"boys({self._print(n)}, {self._print(t)})"

printer = TheochemNumPyPrinter()
printer._module = "np"
printer.known_functions = {k: f"np.{v}" for k, v in _known_functions_numpy.items()}
printer.known_functions["Boys"] = "Boys"
printer.known_constants = {k: f"np.{v}" for k, v in _known_constants_numpy.items()}

In [73]:
def write_eri_module(
    path,
    name="ERI",
    integral_expressions=None,
    parameter_list=None,
    use_cse=True,
):
    lines = [
        "import numpy as np",
        "from .boys import Boys",
        "from numba import njit",
        "@njit(cache = True, fastmath = True)",
        f"def {name}({parameter_list}):",
    ]

    for key, value in integral_expressions.items():
        lines.append(f"    if (i, j, k, l, m, n, o, p, q, r, s, t) == {key}:")

        if use_cse:
            replacements, reduced = sp.cse(
                value,
                symbols=sp.numbered_symbols("t")
            )
            for sym, expr in replacements:
                lines.append(f"        {printer.doprint(sym)} = {printer.doprint(expr)}")
            lines.append(f"        return {printer.doprint(reduced[0])}")
        else:
            lines.append(f"        return {printer.doprint(value)}")

    with open(path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))

In [74]:
path = Path.cwd() / "ERI.py" 

write_eri_module(
    path,
    name="ERI",
    integral_expressions=eri_dict,
    parameter_list=args
)
